In [15]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import box
from shapely.ops import unary_union

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans

from minisom import MiniSom

warnings.filterwarnings("ignore")

In [16]:

# НАСТРОЙКИ

# Где лежат мои данные
BASE_DIR = r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"
SHP_DIR = os.path.join(BASE_DIR, "shp_dbf")



CELL_SIZE = 500 #территория делится на квадраты 500×500 м
RANDOM_STATE = 42

# 12 на 12 нейронов то есть создает 144 типа по признакам клеток с какой либо геологической ситуацией
SOM_X = 12
SOM_Y = 12 

#Количество итераций для обучения
SOM_ITERS = 5000

#сколько групп потом объединяются нейроны SOM.
N_CLUSTERS = 6

OUT_GPKG = os.path.join(BASE_DIR, "som_final_clean.gpkg")
OUT_PNG = os.path.join(BASE_DIR, "som_final_clean.png")


# насколько быстро убывает влияние расстояния до каждого фактора
SCALE_FACIES = 1000
SCALE_PALEO = 1000
SCALE_STRUCT = 900
SCALE_MAGM = 800
SCALE_TECT1 = 800
SCALE_TECT2 = 800


In [17]:
# ФУНКЦИИ

#Собрает полный путь r afqke
def find_file(name):
    return os.path.join(SHP_DIR, name)

# чтобы на вход модели попадали только реальные объекты
def load_layer(path):
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    return gdf

# Приводит все слои к одной системе координат
def to_mask_crs(mask, gdf):
    if mask.crs is not None and gdf.crs is not None and mask.crs != gdf.crs:
        return gdf.to_crs(mask.crs)
    return gdf

#берёт границы маски, создаёт квадраты 500×500,оставляет только те, что пересекают область маски.
def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    minx, miny, maxx, maxy = mask.total_bounds

    xs = np.arange(minx, maxx + cell_size, cell_size)
    ys = np.arange(miny, maxy + cell_size, cell_size)

    cells = []
    ids = []
    cell_id = 0

    for x in xs[:-1]:
        for y in ys[:-1]:
            geom = box(x, y, x + cell_size, y + cell_size)
            if geom.intersects(mask_union):
                cells.append(geom)
                ids.append(cell_id)
                cell_id += 1

    return gpd.GeoDataFrame({"cell_id": ids}, geometry=cells, crs=mask.crs)

#Считает рассторя до какого либо гео слоя
def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    centroids = grid.geometry.centroid
    grid[name] = centroids.apply(lambda pt: pt.distance(source_union)).astype(float)
    return grid

# Преобразует расстояние в степень близости
def proximity(distance, scale):
    d = np.asarray(distance, dtype=float)
    return 1.0 / (1.0 + d / scale)


def normalize_01(arr):
    arr = np.asarray(arr, dtype=float)
    amin = np.nanmin(arr)
    amax = np.nanmax(arr)
    if np.isclose(amax, amin):
        return np.full_like(arr, 0.5, dtype=float)
    return (arr - amin) / (amax - amin)


def collect_points(mask):
    point_layers = []

    for shp in glob.glob(os.path.join(SHP_DIR, "*.shp")):
        gdf = gpd.read_file(shp)
        if gdf.empty:
            continue

        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            gdf = gdf[gdf.geometry.notnull()].copy()
            gdf = gdf[~gdf.geometry.is_empty].copy()
            gdf = to_mask_crs(mask, gdf)
            gdf = gdf[gdf.geometry.intersects(unary_union(mask.geometry))].copy()
            if len(gdf) > 0:
                point_layers.append(gdf)

    if len(point_layers) == 0:
        return None

    points = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(points, geometry="geometry", crs=mask.crs)

In [18]:
#Загрузка
# mask — область прогноза;
# facies — литолого-фациальный фактор
# tect1, tect2 — разломы
# paleo — палеогеоморфология
# struct — структурный фактор
# magm — дайки
# points — точки проявлений

mask = load_layer(find_file("svita_new.shp"))
facies = load_layer(find_file("fasii.shp"))
tect1 = load_layer(find_file("glub_raz_nw.shp"))
tect2 = load_layer(find_file("glub_r_nw.shp"))
paleo = load_layer(find_file("gr_dol_vp_poly.shp"))
struct = load_layer(find_file("kory.shp"))
magm = load_layer(find_file("dayki_buf.shp"))

facies = to_mask_crs(mask, facies)
tect1 = to_mask_crs(mask, tect1)
tect2 = to_mask_crs(mask, tect2)
paleo = to_mask_crs(mask, paleo)
struct = to_mask_crs(mask, struct)
magm = to_mask_crs(mask, magm)

points = collect_points(mask)


In [19]:
# СЕТКА
grid = build_grid(mask, CELL_SIZE)

In [20]:
# ПРИЗНАКИ

# Сначала считаются расстояния:
# dist_facies
# dist_paleo
# dist_struct
# dist_magm
# dist_tect1
# dist_tect2
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

# Потом расстояния превращаются в близости:
# prox_facies
# prox_paleo
grid["prox_facies"] = proximity(grid["dist_facies"], SCALE_FACIES)
grid["prox_paleo"] = proximity(grid["dist_paleo"], SCALE_PALEO)
grid["prox_struct"] = proximity(grid["dist_struct"], SCALE_STRUCT)
grid["prox_magm"] = proximity(grid["dist_magm"], SCALE_MAGM)
grid["prox_tect1"] = proximity(grid["dist_tect1"], SCALE_TECT1)
grid["prox_tect2"] = proximity(grid["dist_tect2"], SCALE_TECT2)

#  если ячейка близка и к разлому, и к дайке, это моолько к одному объекжет быть важнее, чем близость к одному какому нибудь обьекту 
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = ((grid["prox_tect1"] + grid["prox_tect2"]) / 2.0) * grid["prox_magm"]
grid["tect_struct_intersection"] = ((grid["prox_tect1"] + grid["prox_tect2"]) / 2.0) * grid["prox_struct"]
grid["paleo_struct_intersection"] = grid["prox_paleo"] * grid["prox_struct"]


# геологическая оценк. Используется как дополнительный признак
grid["geo_score_raw"] = (
    1.4 * grid["prox_tect1"] +
    1.4 * grid["prox_tect2"] +
    1.2 * grid["prox_magm"] +
    1.1 * grid["prox_struct"] +
    1.0 * grid["prox_paleo"] +
    0.9 * grid["prox_facies"] +
    0.5 * grid["tect_intersection"] +
    0.4 * grid["tect_magm_intersection"] +
    0.3 * grid["tect_struct_intersection"] +
    0.2 * grid["paleo_struct_intersection"]
)
grid["geo_score"] = 0.7 * normalize_01(grid["geo_score_raw"]) + 0.3

feature_cols = [
    "prox_facies",
    "prox_paleo",
    "prox_struct",
    "prox_magm",
    "prox_tect1",
    "prox_tect2",
    "tect_intersection",
    "tect_magm_intersection",
    "tect_struct_intersection",
    "paleo_struct_intersection",
    "geo_score",
]

X = grid[feature_cols].fillna(0).copy()


In [21]:
# TARGET

#есть здесь руда или нет по
# шлиховые ареалы золота и урана
# литохимические ареалы золота и урана
# рудопроявления
# пункты минерализации

grid["target"] = 0

use_supervised = False

if points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(
            points,
            grid[["cell_id", "geometry"]],
            how="left",
            predicate="within"
        )
    except Exception:
        joined = gpd.sjoin(
            points,
            grid[["cell_id", "geometry"]],
            how="left",
            op="within"
        )

    positive_cells = joined["cell_id"].dropna().unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "target"] = 1

    if grid["target"].sum() >= 5 and grid["target"].sum() < len(grid):
        use_supervised = True

In [22]:
# LOGISTIC REGRESSION

#автоматическое определение значимости факторов если обучаться не на чем модель использует экспертную геологическую оценку
if use_supervised:
    lr = LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=2000,
        class_weight="balanced"
    )
    lr.fit(X, grid["target"])
    grid["ml_score"] = normalize_01(lr.predict_proba(X)[:, 1])
else:
    grid["ml_score"] = grid["geo_score"]

In [23]:
# SOM
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# получает на вход признаки ячеек
# группирует похожие ячейки
# отображает многомерные данные в компактную двумерную структуру
som = MiniSom(
    x=SOM_X,
    y=SOM_Y,
    input_len=X_scaled.shape[1],
    sigma=1.3,
    learning_rate=0.45,
    random_seed=RANDOM_STATE
)

som.random_weights_init(X_scaled)
som.train_random(X_scaled, SOM_ITERS)
#lля каждой ячейки определяется к какому относится нейрону SOM 
winners = np.array([som.winner(x) for x in X_scaled])
grid["som_x"] = winners[:, 0]
grid["som_y"] = winners[:, 1]
grid["som_node"] = grid["som_x"].astype(str) + "_" + grid["som_y"].astype(str)

#yейроны SOM дополнительно объединяются в несколько кластеров.
som_weights = som.get_weights().reshape(SOM_X * SOM_Y, X_scaled.shape[1])
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
neuron_cluster = kmeans.fit_predict(som_weights)

node_to_cluster = {}
idx = 0
for i in range(SOM_X):
    for j in range(SOM_Y):
        node_to_cluster[f"{i}_{j}"] = int(neuron_cluster[idx])
        idx += 1

grid["cluster"] = grid["som_node"].map(node_to_cluster).astype(int)


In [24]:
# CLUSTER SCORE

# для каждого кластера оценивается, насколько он перспективен.
#tсли ячейка попала в хороший кластер, это повышает её перспективность.
cluster_geo = grid.groupby("cluster")["geo_score"].mean().reset_index(name="cluster_geo_mean")
cluster_ml = grid.groupby("cluster")["ml_score"].mean().reset_index(name="cluster_ml_mean")
cluster_stats = cluster_geo.merge(cluster_ml, on="cluster", how="outer")

if use_supervised:
    cluster_hits = (
        grid.groupby("cluster")
        .agg(cells=("cell_id", "count"), positives=("target", "sum"))
        .reset_index()
    )
    cluster_hits["hit_rate"] = cluster_hits["positives"] / cluster_hits["cells"]

    cluster_stats = cluster_stats.merge(cluster_hits, on="cluster", how="left")
    cluster_stats["hit_rate"] = cluster_stats["hit_rate"].fillna(0)

    cluster_stats["cluster_score"] = normalize_01(
        0.50 * cluster_stats["cluster_ml_mean"] +
        0.30 * cluster_stats["cluster_geo_mean"] +
        0.20 * normalize_01(cluster_stats["hit_rate"])
    )
else:
    cluster_stats["cluster_score"] = normalize_01(
        0.60 * cluster_stats["cluster_ml_mean"] +
        0.40 * cluster_stats["cluster_geo_mean"]
    )

cluster_score_map = dict(zip(cluster_stats["cluster"], cluster_stats["cluster_score"]))
grid["cluster_score"] = grid["cluster"].map(cluster_score_map).astype(float)

In [25]:
# ИТОГ
#Состоит из
#ml_score — что узнала модель по точкам
# cluster_score — что показала SOM
# geo_score — экспертная геология

#prospectivity это главный итоговый показатель проекта
# Чем выше значение, тем выше перспективность ячейки
if use_supervised:
    grid["prospectivity_raw"] = (
        0.55 * grid["ml_score"] +
        0.25 * grid["cluster_score"] +
        0.20 * grid["geo_score"]
    )
else:
    grid["prospectivity_raw"] = (
        0.50 * grid["cluster_score"] +
        0.50 * grid["geo_score"]
    )

grid["local_bonus"] = normalize_01(
    0.4 * grid["tect_intersection"] +
    0.35 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"]
)

grid["prospectivity_raw"] += 0.08 * grid["local_bonus"]
grid["prospectivity"] = normalize_01(grid["prospectivity_raw"])

# Разбивает итог на 5 классов:
# very_low
# low
# medium
# high
# very_high
grid["prospect_class"] = pd.qcut(
    grid["prospectivity"],
    q=5,
    labels=["very_low", "low", "medium", "high", "very_high"],
    duplicates="drop"
)

# Выделяет 10% лучших ячеек.
threshold = grid["prospectivity"].quantile(0.90)
grid["top10"] = (grid["prospectivity"] >= threshold).astype(int)



# СОХРАНЕНИЕ

if os.path.exists(OUT_GPKG):
    try:
        os.remove(OUT_GPKG)
    except Exception:
        pass

grid.to_file(OUT_GPKG, layer="som_final_clean", driver="GPKG")

if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

top_grid = grid[grid["top10"] == 1].copy()
if len(top_grid) > 0:
    top_grid.to_file(OUT_GPKG, layer="top10_zones", driver="GPKG")



# ВИЗУАЛИЗАЦИЯ

fig, ax = plt.subplots(figsize=(12, 12))

grid.plot(
    column="prospectivity",
    ax=ax,
    legend=True,
    cmap="RdYlBu_r",
    linewidth=0
)

mask.boundary.plot(ax=ax, color="black", linewidth=0.6)

if points is not None and len(points) > 0:
    points.plot(
        ax=ax,
        color="yellow",
        markersize=12,
        edgecolor="black",
        linewidth=0.3
    )

ax.set_title("Карта")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300)
plt.close()

